## **Hyperparameters of Naive Bayes in Scikit-Learn**

### **Topic Roadmap**

**1. Environment Setup and Data Loading**

**2. The `alpha` Hyperparameter (Laplace Smoothing)**

**3. The `fit_prior` and `class_prior` Hyperparameters**

**4. The `var_smoothing` Hyperparameter (Gaussian NB)**

**5. The `binarize` Hyperparameter (Bernoulli NB)**

### **1. Environment Setup and Data Loading**

Import standard libraries and generate synthetic datasets. We will create a discrete dataset for Multinomial/Bernoulli NB and a continuous dataset for Gaussian NB.

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification

from sklearn.naive_bayes import (
    GaussianNB,
    MultinomialNB,
    BernoulliNB
)

In [2]:
# Dataset for Gaussian NB (Continuous Features)
X_cont, y_cont = make_classification(n_samples=1000, n_features=5, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cont, y_cont, test_size=0.2, random_state=42)

# Dataset for Multinomial/Bernoulli NB (Discrete/Count Features)
np.random.seed(42)
X_disc = np.random.randint(0, 10, size=(1000, 5))
y_disc = np.random.randint(0, 2, size=1000)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_disc, y_disc, test_size=0.2, random_state=42)

### **2. The `alpha` Hyperparameter (Laplace Smoothing)**

Available in `MultinomialNB`, `BernoulliNB`, and `CategoricalNB`. It adds a smoothing factor to prevent zero probabilities when a feature category is missing in the training data but appears in the test data.

**Values:**
*   `alpha = 1.0` (Default): Laplace smoothing.
*   `0 < alpha < 1`: Lidstone smoothing.
*   `alpha = 0`: No smoothing (can lead to zero-frequency errors).

In [3]:
# Standard Laplace Smoothing
mnb_laplace = MultinomialNB(alpha=1.0)
mnb_laplace.fit(X_train_d, y_train_d)
acc_laplace = accuracy_score(y_test_d, mnb_laplace.predict(X_test_d))

In [4]:
# Lidstone Smoothing (weaker smoothing)
mnb_lidstone = MultinomialNB(alpha=0.1)
mnb_lidstone.fit(X_train_d, y_train_d)
acc_lidstone = accuracy_score(y_test_d, mnb_lidstone.predict(X_test_d))

print(f"Laplace Smoothing (alpha=1.0) Accuracy:  {acc_laplace:.4f}")
print(f"Lidstone Smoothing (alpha=0.1) Accuracy: {acc_lidstone:.4f}")

Laplace Smoothing (alpha=1.0) Accuracy:  0.4900
Lidstone Smoothing (alpha=0.1) Accuracy: 0.4900


### **3. The `fit_prior` and `class_prior` Hyperparameters**

**`fit_prior` (bool):** 
Determines whether to learn class prior probabilities from the data. If set to `False`, a uniform prior will be used.

**`class_prior` (array-like):** 
Allows you to manually specify the prior probabilities of the classes. If specified, the priors are not adjusted according to the data.

In [5]:
# Do not learn from data; assume uniform distribution (50/50 for binary)
mnb_uniform = MultinomialNB(fit_prior=False)
mnb_uniform.fit(X_train_d, y_train_d)

# Manually set priors (e.g., heavily biasing the model toward class 1)
mnb_custom_prior = MultinomialNB(class_prior=[0.1, 0.9])
mnb_custom_prior.fit(X_train_d, y_train_d)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.","[0.1, 0.9]"


In [6]:
print(f"Uniform Prior Accuracy: {accuracy_score(y_test_d, mnb_uniform.predict(X_test_d)):.4f}")
print(f"Custom Prior Accuracy:  {accuracy_score(y_test_d, mnb_custom_prior.predict(X_test_d)):.4f}")

Uniform Prior Accuracy: 0.4800
Custom Prior Accuracy:  0.4950


### **4. The `var_smoothing` Hyperparameter (Gaussian NB)**

Exclusive to `GaussianNB`. It is a stability calculation factor representing the portion of the largest variance of all features that is added to variances.

Increasing it can artificially widen the Gaussian curve, which smooths out the model and can act as a form of regularization.

In [7]:
# Default var_smoothing is 1e-9
gnb_default = GaussianNB(var_smoothing=1e-9)
gnb_default.fit(X_train_c, y_train_c)

# Increased var_smoothing to act as heavy regularization
gnb_smoothed = GaussianNB(var_smoothing=1e-1)
gnb_smoothed.fit(X_train_c, y_train_c)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",0.1


In [8]:
print(f"Default var_smoothing Accuracy:  {accuracy_score(y_test_c, gnb_default.predict(X_test_c)):.4f}")
print(f"Increased var_smoothing Accuracy:{accuracy_score(y_test_c, gnb_smoothed.predict(X_test_c)):.4f}")

Default var_smoothing Accuracy:  0.8600
Increased var_smoothing Accuracy:0.8550


### **5. The `binarize` Hyperparameter (Bernoulli NB)**

Exclusive to `BernoulliNB`. It defines a threshold for mapping continuous or count-based sample features to booleans (1 or 0). If `None`, input is presumed to already consist of binary vectors.

In [9]:
# Features > 2.0 become 1 (True), features <= 2.0 become 0 (False)
bnb_binarized = BernoulliNB(binarize=2.0)
bnb_binarized.fit(X_train_d, y_train_d)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",2.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [10]:
print(f"Bernoulli NB (Threshold=2.0) Accuracy: {accuracy_score(y_test_d, bnb_binarized.predict(X_test_d)):.4f}")

Bernoulli NB (Threshold=2.0) Accuracy: 0.4450


### **Key Revision Notes**

- **`alpha`:** The single most important hyperparameter to tune for discrete Naive Bayes algorithms. Prevents the "Zero Probability Problem." Tuning it is the primary way to optimize a `MultinomialNB` model.
- **`class_prior` vs. `fit_prior`:** If you know the true distribution of your target population is different from your training set, use `class_prior` to inject that domain knowledge.
- **`var_smoothing`:** The primary hyperparameter to tune in `GaussianNB`. Useful when the dataset has highly correlated features or is noisy.
- **`binarize`:** Automates feature extraction inside the model, turning frequency counts into mere presence/absence indicators on the fly.